This notebook is to prep for my interview with an undisclosed company in Adtech
- Date: June 19th, 2026

# Prep
## Imports

In [2]:
## if running into errors with pip, add to PATH
# import sys, os
# print(os.path.dirname(sys.executable))


In [3]:
!pip install -r ../requirements.txt
# !python -m pip freeze > requirements.txt


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import sklearn
import seaborn as sns
import scipy
# import scikit-learn

## Constants

In [5]:
DATA_PATH = "../data/"

## Load in Data

In [6]:
campaign_df = pd.read_csv(DATA_PATH + "campaign_performance.csv")
print(campaign_df.shape[0])
campaign_df.head(3)

4288


,date,campaign_id,campaign_name,channel,region,weather_severity_index,impressions,clicks,spend_cad,conversions,revenue_cad,cpm_cad,cpc_cad,roas
0,2025-12-22,CMP1002,Premium_Upgrade_Q1,Connected TV,Quebec,0.871,48337,951.5,702.84,22.10,1481.14,14.54,0.739,2.107
1,2025-11-29,CMP1004,Brand_Lift_National,Connected TV,Quebec,0.374,36764,539.9,248.47,25.07,1746.93,6.76,0.460,7.031
2,2025-12-11,CMP1006,Spring_Forecast_Launch,Display,British Columbia,0.982,83195,1353.7,759.46,56.94,2276.05,9.13,0.561,2.997


In [7]:
users_df = pd.read_csv(DATA_PATH + "users.csv")
print(users_df.shape[0])
users_df.head(3)

3010


,user_id,signup_date,acquisition_channel,region,plan_type,avg_monthly_sessions,alerts_enabled,push_notifications_enabled,tenure_months,churned,churn_date,ltv_cad
0,U100452,2024-12-10,Native,Atlantic,Free,0.85,1,0,12.90,0,NaN,0.0
1,U101657,2025-11-24,Referral,Ontario,Free,5.90,1,1,1.27,0,NaN,0.0
2,U100062,2024-12-03,Organic,Ontario,Premium,0.27,0,0,13.13,0,NaN,304.5


In [8]:
adtech_df = pd.read_excel(DATA_PATH + "adtech_practice_dataset.xlsx")
print(adtech_df.shape[0])
adtech_df.head(3)

4288


,date,campaign_id,campaign_name,channel,region,weather_severity_index,impressions,clicks,spend_cad,conversions,revenue_cad,cpm_cad,cpc_cad,roas
0,2025-12-22,CMP1002,Premium_Upgrade_Q1,Connected TV,Quebec,0.871,48337,951.5,702.84,22.10,1481.14,14.54,0.739,2.107
1,2025-11-29,CMP1004,Brand_Lift_National,Connected TV,Quebec,0.374,36764,539.9,248.47,25.07,1746.93,6.76,0.460,7.031
2,2025-12-11,CMP1006,Spring_Forecast_Launch,Display,British Columbia,0.982,83195,1353.7,759.46,56.94,2276.05,9.13,0.561,2.997


# Day 1 analysis
- Derive Adtech metrics (Revenue, CPM, CPC, ROAS) 
- Compare to precalculated ones (revenue_cad, cpm_cad, cpc_cad, roas)
- EDA

In [9]:
campaign_base_df = campaign_df[['date', 'impressions', 'clicks', 'spend_cad', 'conversions', 'campaign_id', 'campaign_name', 'channel', 'region', 'weather_severity_index']]

campaign_base_df.head(1)

,date,impressions,clicks,spend_cad,conversions,campaign_id,campaign_name,channel,region,weather_severity_index
0,2025-12-22,48337,951.5,702.84,22.1,CMP1002,Premium_Upgrade_Q1,Connected TV,Quebec,0.871


## 1.1 Understanding the data, what I'm being given, some context.

In [15]:
campaign_base_df.describe()

,impressions,clicks,spend_cad,conversions,weather_severity_index
count,4.288000e+03,4245.000000,4267.000000,4288.000000,4288.000000
mean,6.014057e+04,903.015940,551.677874,36.409611,0.605392
std,6.635299e+04,544.908681,471.167486,27.569856,0.228200
min,5.000000e+02,6.400000,-542.140000,0.230000,0.105000
25%,4.366350e+04,539.900000,321.355000,17.915000,0.405000
50%,5.532200e+04,810.100000,457.200000,30.230000,0.632000
75%,6.743800e+04,1138.500000,624.240000,47.765000,0.781000
max,2.125964e+06,5426.700000,6599.410000,357.890000,1.000000


- Good to note how some of the data ranges, like Clicks varies from ~6 - ~5400, vs Weather is 0.10 - 1.00.
- Knowing the data format helps us analyze it accordingly, like weather is a percentage.

In [13]:
campaign_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4288 entries, 0 to 4287
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    4288 non-null   str    
 1   campaign_id             4288 non-null   str    
 2   campaign_name           4288 non-null   str    
 3   channel                 4288 non-null   str    
 4   region                  4254 non-null   str    
 5   weather_severity_index  4288 non-null   float64
 6   impressions             4288 non-null   int64  
 7   clicks                  4245 non-null   float64
 8   spend_cad               4267 non-null   float64
 9   conversions             4288 non-null   float64
 10  revenue_cad             4288 non-null   float64
 11  cpm_cad                 4288 non-null   float64
 12  cpc_cad                 4203 non-null   float64
 13  roas                    4288 non-null   float64
 14  CPM                     4267 non-null   float64
 15

In [14]:
users_df.describe()

,avg_monthly_sessions,alerts_enabled,push_notifications_enabled,tenure_months,churned,ltv_cad
count,2980.000000,3010.000000,3010.000000,3010.00000,3010.000000,2995.000000
mean,7.088003,0.713621,0.544186,10.11986,0.229900,76.230317
std,5.287196,0.452144,0.498127,5.33177,0.420838,130.630449
min,0.100000,0.000000,0.000000,1.07000,0.000000,0.000000
25%,3.310000,0.000000,0.000000,5.50000,0.000000,0.000000
50%,5.930000,1.000000,1.000000,9.97000,0.000000,0.000000
75%,9.310000,1.000000,1.000000,14.76000,0.000000,115.365000
max,50.760000,1.000000,1.000000,19.30000,1.000000,1015.600000


## 1.2 Make Adtech metrics

In [17]:
# # impressions, clicks, spend_cad, conversions
# # Revenue =
# # campaign_df['Revenue CAD'] = campaign_base_df[]

# # CPM = Cost per 1000 impressions, so CPM = spend / impressions * 1000
campaign_df['CPM'] = (campaign_base_df['spend_cad'] / campaign_base_df['impressions']) * 1000

# CPC - Cost per click, so CPC = spend / clicks 
campaign_df['CPC'] = campaign_base_df['spend_cad'] / campaign_base_df['clicks']

# # ROAS - Return on Ad Spend, so ROAS = conversions - spend 
# campaign_df['ROAS'] = campaign_base_df['spend_cad'] / campaign_base_df['conversions']

campaign_df.head(3)


,date,campaign_id,campaign_name,channel,region,weather_severity_index,impressions,clicks,spend_cad,conversions,revenue_cad,cpm_cad,cpc_cad,roas,CPM,CPC,ROAS
0,2025-12-22,CMP1002,Premium_Upgrade_Q1,Connected TV,Quebec,0.871,48337,951.5,702.84,22.10,1481.14,14.54,0.739,2.107,14.540414,0.738665,31.802715
1,2025-11-29,CMP1004,Brand_Lift_National,Connected TV,Quebec,0.374,36764,539.9,248.47,25.07,1746.93,6.76,0.460,7.031,6.758514,0.460215,9.911049
2,2025-12-11,CMP1006,Spring_Forecast_Launch,Display,British Columbia,0.982,83195,1353.7,759.46,56.94,2276.05,9.13,0.561,2.997,9.128674,0.561025,13.337900
